# FIFA World Cup 2026 — Matches Data Collection

This notebook collects the FIFA World Cup 2026 match data from the official FIFA API.

The dataset will contain the 104 tournament matches, including match date, stage, teams, stadium, scores, winner, and result information.

Source:
Official FIFA World Cup 2026 API.

In [1]:
import requests
import pandas as pd
import re

print("All libraries imported successfully!")

All libraries imported successfully!


## 2. FIFA Matches API Connection

We will connect to FIFA's public API endpoint for World Cup match data.

Competition ID: 17

Season ID: 285023

Stage ID: 289273

The API response will be inspected before we extract the 104 matches.

In [25]:
matches_api_url = (
    "https://cxm-api.fifa.com/fifaplusweb/api/pages/en/"
    "tournaments/mens/worldcup/canadamexicousa2026/"
    "articles/match-schedule-fixtures-results-teams-stadiums"
)

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/139.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json"
}

response = requests.get(
    matches_api_url,
    headers=headers,
    timeout=30
)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Response size:", len(response.content), "bytes")

Status code: 200
Content type: application/json; charset=utf-8
Response size: 4448 bytes


## 3. Inspect FIFA Schedule API Response

Inspect the structure of the official FIFA World Cup 2026
schedule response before extracting any match data.

In [26]:
matches_page_data = response.json()

print("Top-level keys:")
print(matches_page_data.keys())

print("\nNumber of sections:")
print(len(matches_page_data.get("sections", [])))

print("\nSections:")
for section in matches_page_data.get("sections", []):
    print(section)

Top-level keys:
dict_keys(['pageId', 'internalTitle', 'template', 'createdDate', 'tags', 'relativeUrl', 'relativeUrlsSEO', 'language', 'meta', 'sections', 'backgroundCSS', 'headerBehavior', 'sectionsVertical', 'sectionsHorizontal', 'theme', 'customTheme', 'tournamentState'])

Number of sections:
4

Sections:
{'entryId': 'S9YG2JmeGYaMUCBbm0CcD', 'entryType': 'article', 'properties': {}, 'entryEndpoint': '/sections/article/S9YG2JmeGYaMUCBbm0CcD?locale=en'}
{'entryId': '6A5AzKAdAkbIWdl2Os1i8n', 'entryType': 'sectionPromoCarousel', 'properties': {}, 'entryEndpoint': 'sections/promoCarousel/6A5AzKAdAkbIWdl2Os1i8n?locale=en'}
{'entryId': '4TQNjIzRzQW638tY5jAcil', 'entryType': 'sectionPromoCarousel', 'properties': {}, 'entryEndpoint': 'sections/promoCarousel/4TQNjIzRzQW638tY5jAcil?locale=en'}
{'entryId': '39WGWL4DARJ5X86DGTPja0', 'entryType': 'sectionPromoCarousel', 'properties': {}, 'entryEndpoint': 'sections/promoCarousel/39WGWL4DARJ5X86DGTPja0?locale=en'}


## 4. Retrieve the FIFA World Cup 2026 Schedule Article

The page response references the main article section through
an `entryEndpoint`.

We will retrieve this section to inspect the actual
World Cup 2026 schedule content.

In [27]:
# Build the official FIFA article-section API URL

article_url = (
    "https://cxm-api.fifa.com/fifaplusweb/api/"
    "sections/article/S9YG2JmeGYaMUCBbm0CcD?locale=en"
)

article_response = requests.get(
    article_url,
    headers=headers,
    timeout=30
)

print("Status code:", article_response.status_code)
print("Content type:", article_response.headers.get("Content-Type"))
print("Response size:", len(article_response.content), "bytes")

Status code: 200
Content type: application/json; charset=utf-8
Response size: 133269 bytes


## 5. Inspect FIFA Article Content

Inspect the structure of the FIFA World Cup 2026 article section
to identify where the official match schedule and results are stored.

In [28]:
# Parse the FIFA article section response

article_data = article_response.json()

print("Top-level keys:")
print(article_data.keys())

print("\nNumber of top-level content items:")

for key, value in article_data.items():
    if isinstance(value, list):
        print(f"{key}: {len(value)} items")
    elif isinstance(value, dict):
        print(f"{key}: dictionary")
    else:
        print(f"{key}: {type(value).__name__}")

Top-level keys:
dict_keys(['entryType', 'entryId', 'articleTitle', 'articlePublishedDate', 'heroImage', 'richtext', 'publishedLabel', 'backLabel', 'shareLabel', 'authorLabel', 'theme', 'customTheme', 'articlePreviewText', 'previewImage', 'articlePageInternalTitle'])

Number of top-level content items:
entryType: str
entryId: str
articleTitle: str
articlePublishedDate: str
heroImage: dictionary
richtext: dictionary
publishedLabel: str
backLabel: str
shareLabel: str
authorLabel: str
theme: str
customTheme: dictionary
articlePreviewText: str
previewImage: dictionary
articlePageInternalTitle: str


## 6. Inspect FIFA Richtext Structure

The FIFA article stores its schedule content inside the `richtext`
object.

Inspect its structure to determine how the 104 matches are represented.

In [29]:
richtext_data = article_data["richtext"]

print("Richtext type:", type(richtext_data))

print("\nRichtext keys:")
print(richtext_data.keys())

for key, value in richtext_data.items():
    if isinstance(value, list):
        print(f"\n{key}: list with {len(value)} items")
    elif isinstance(value, dict):
        print(f"\n{key}: dictionary")
    else:
        print(f"\n{key}: {type(value).__name__}")

Richtext type: <class 'dict'>

Richtext keys:
dict_keys(['nodeType', 'data', 'content'])

nodeType: str

data: dictionary

content: list with 133 items


## 7. Inspect FIFA Schedule Content Blocks

Inspect the 133 content blocks inside the FIFA article to identify
the blocks containing the official World Cup 2026 match schedule.

In [30]:
# Inspect the node types used in the FIFA schedule article

content_blocks = richtext_data["content"]

node_types = {}

for block in content_blocks:
    node_type = block.get("nodeType", "unknown")
    node_types[node_type] = node_types.get(node_type, 0) + 1

print("Node types found:")
for node_type, count in node_types.items():
    print(f"{node_type}: {count}")

Node types found:
paragraph: 39
heading-3: 10
embedded-entry-block: 42
hr: 8
heading-4: 34


## 8. Inspect Embedded Schedule Blocks

Inspect the embedded entries in the FIFA article to determine
whether they contain the official World Cup 2026 match data.

In [31]:
# Inspect the structure of embedded-entry blocks

embedded_blocks = [
    block
    for block in content_blocks
    if block.get("nodeType") == "embedded-entry-block"
]

print("Embedded blocks found:", len(embedded_blocks))

for i, block in enumerate(embedded_blocks[:5]):
    print(f"\n--- Embedded block {i + 1} ---")
    print("Keys:", block.keys())
    print("Data:", block.get("data"))

Embedded blocks found: 42

--- Embedded block 1 ---
Keys: dict_keys(['nodeType', 'content', 'data'])
Data: {'target': {'geolocalisation': [], 'contentTypesCheckboxValue': 'Promotional External Link', 'link': 'https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026/tickets', 'internalTitle': 'Promotional External Link > FWC 2026 > Explore Ticket Options > Article Embed', 'title': 'Tickets - FIFA World Cup 26™', 'subtitle': 'Start your FIFA World Cup 26™ journey by exploring our ticket options!', 'buttonLabel': 'Explore details', 'target': 'New window', 'semanticTags': [], 'semanticTagReferences': [], 'primaryRoofline': [], 'primaryRooflineReference': [], 'secondaryRoofline': [], 'secondaryRooflineReference': [], 'appShareButton': False, 'appPullToRefresh': False, 'openInApp': True, 'sys': {'locale': 'en', 'contentType': {'systemProperties': {'id': 'promotionalExternalLink'}}, 'id': '5X9fOv7i0QsBb6DpiI9agc', 'createdAt': '2025-09-15T12:58:15.641Z', 'updatedAt': '2026-03-30T

## 8. Inspect FIFA Schedule Text and Headings

Inspect the headings and paragraphs in the FIFA World Cup 2026
article to identify the section containing the official match schedule.

In [32]:
# Inspect headings and paragraphs from the FIFA article

text_blocks = []

for block in content_blocks:
    node_type = block.get("nodeType")

    if node_type in ["heading-3", "heading-4", "paragraph"]:
        text = ""

        for item in block.get("content", []):
            if isinstance(item, dict):
                text += item.get("value", "")

        text_blocks.append({
            "nodeType": node_type,
            "text": text.strip()
        })

text_blocks_df = pd.DataFrame(text_blocks)

display(text_blocks_df)

,nodeType,text
0,paragraph,"On this page you'll find the full schedule, i..."
1,paragraph,You can also view details of all World Cup 202...
2,paragraph,From the tournament's first whistle on 11 June...
3,heading-3,
4,heading-3,
...,...,...
78,paragraph,Match 103 – - Miami Stadium
79,heading-3,FIFA World Cup 26 final
80,heading-4,"Sunday, 19 July 2026"
81,paragraph,Match 104 – - New York New Jersey Stadium


## 9. Inspect FIFA Match 1 Structure

Inspect the richtext block containing Match 1 to determine how
FIFA stores the teams, date, and stadium information.

In [33]:
# Find the richtext blocks containing Match 1

for i, block in enumerate(content_blocks):
    block_text = ""

    for item in block.get("content", []):
        if isinstance(item, dict):
            block_text += str(item.get("value", ""))

    if "Match 1" in block_text:
        print("Match 1 found in block:", i)
        print("\nNode type:", block.get("nodeType"))
        print("\nFull block structure:")
        print(block)

Match 1 found in block: 112

Node type: paragraph

Full block structure:
{'nodeType': 'paragraph', 'data': {}, 'content': [{'nodeType': 'text', 'data': {}, 'value': 'Match 99 – ', 'marks': []}, {'nodeType': 'hyperlink', 'data': {'uri': 'https://www.fifa.com/en/match-centre/match/17/285023/289289/400021539'}, 'content': [{'nodeType': 'text', 'data': {}, 'value': 'Norway 1-2 England (AET)', 'marks': []}]}, {'nodeType': 'text', 'data': {}, 'value': ' - Miami Stadium\nMatch 100 – ', 'marks': []}, {'nodeType': 'hyperlink', 'data': {'uri': 'https://www.fifa.com/en/match-centre/match/17/285023/289289/400021537'}, 'content': [{'nodeType': 'text', 'data': {}, 'value': 'Argentina 3-1 Switzerland (AET)', 'marks': []}]}, {'nodeType': 'text', 'data': {}, 'value': ' - Kansas City Stadium', 'marks': []}]}
Match 1 found in block: 118

Node type: paragraph

Full block structure:
{'nodeType': 'paragraph', 'data': {}, 'content': [{'nodeType': 'text', 'data': {}, 'value': 'Match 101 – ', 'marks': []}, {'n

## 10. Identify the 104 FIFA Match Links

Extract all hyperlinks from the FIFA article that point to
FIFA World Cup 2026 match-centre pages.

The expected number of matches is 104.

In [34]:
# Find all World Cup 2026 match-centre links

match_links = []

for block in content_blocks:
    for item in block.get("content", []):
        if item.get("nodeType") == "hyperlink":
            uri = item.get("data", {}).get("uri", "")
            
            if "/en/match-centre/match/17/285023/" in uri:
                match_links.append({
                    "url": uri,
                    "text": "".join(
                        child.get("value", "")
                        for child in item.get("content", [])
                        if child.get("nodeType") == "text"
                    )
                })

print("World Cup 2026 match links found:", len(match_links))

if match_links:
    print("\nFirst 5 match links:")
    for match in match_links[:5]:
        print(match)

    print("\nLast 5 match links:")
    for match in match_links[-5:]:
        print(match)

World Cup 2026 match links found: 104

First 5 match links:
{'url': 'https://www.fifa.com/en/match-centre/match/17/285023/289273/400021443', 'text': 'Mexico 2-0 South Africa'}
{'url': 'https://www.fifa.com/en/match-centre/match/17/285023/289273/400021441', 'text': 'Korea Republic 2-1 Czechia'}
{'url': 'https://www.fifa.com/en/match-centre/match/17/285023/289273/400021449', 'text': 'Canada 1-1 Bosnia and Herzegovina'}
{'url': 'https://www.fifa.com/en/match-centre/match/17/285023/289273/400021458', 'text': 'USA 4-1 Paraguay'}
{'url': 'https://www.fifa.com/en/match-centre/match/17/285023/289273/400021453', 'text': 'Haiti 0-1 Scotland'}

Last 5 match links:
{'url': 'https://www.fifa.com/en/match-centre/match/17/285023/289289/400021537', 'text': 'Argentina 3-1 Switzerland (AET)'}
{'url': 'https://www.fifa.com/en/match-centre/match/17/285023/289290/400021541', 'text': 'France 0-2 Spain'}
{'url': 'https://www.fifa.com/en/match-centre/match/17/285023/289290/400021540', 'text': 'England 1-2 Arg

## 11. Create the FIFA Match Links Dataset

Create a structured DataFrame containing the 104 official FIFA
World Cup 2026 match-centre links and their displayed match results.

These links will be used to retrieve the detailed information
for each match.

In [35]:
# Create a DataFrame containing all 104 FIFA match links

match_links_df = pd.DataFrame(match_links)

print("Rows:", len(match_links_df))
print("Columns:", list(match_links_df.columns))

display(match_links_df.head())

Rows: 104
Columns: ['url', 'text']


,url,text
0,https://www.fifa.com/en/match-centre/match/17/...,Mexico 2-0 South Africa
1,https://www.fifa.com/en/match-centre/match/17/...,Korea Republic 2-1 Czechia
2,https://www.fifa.com/en/match-centre/match/17/...,Canada 1-1 Bosnia and Herzegovina
3,https://www.fifa.com/en/match-centre/match/17/...,USA 4-1 Paraguay
4,https://www.fifa.com/en/match-centre/match/17/...,Haiti 0-1 Scotland


## 12. Extract FIFA Match IDs

Extract the FIFA competition, season, stage, and match IDs
from the official FIFA match-centre URLs.

These IDs will allow us to retrieve structured information
for each of the 104 matches.

In [37]:
# Extract FIFA IDs from each match-centre URL

match_links_df[["CompetitionID", "SeasonID", "StageID", "MatchID"]] = (
    match_links_df["url"]
    .str.extract(
        r"/match/(\d+)/(\d+)/(\d+)/(\d+)"
    )
)

# Convert IDs from text to integers
id_columns = ["CompetitionID", "SeasonID", "StageID", "MatchID"]

for column in id_columns:
    match_links_df[column] = pd.to_numeric(
        match_links_df[column],
        errors="raise"
    )

print("Rows:", len(match_links_df))
print("\nColumns:")
print(match_links_df.columns.tolist())

display(match_links_df.head())

Rows: 104

Columns:
['url', 'text', 'CompetitionID', 'SeasonID', 'StageID', 'MatchID']


,url,text,CompetitionID,SeasonID,StageID,MatchID
0,https://www.fifa.com/en/match-centre/match/17/...,Mexico 2-0 South Africa,17,285023,289273,400021443
1,https://www.fifa.com/en/match-centre/match/17/...,Korea Republic 2-1 Czechia,17,285023,289273,400021441
2,https://www.fifa.com/en/match-centre/match/17/...,Canada 1-1 Bosnia and Herzegovina,17,285023,289273,400021449
3,https://www.fifa.com/en/match-centre/match/17/...,USA 4-1 Paraguay,17,285023,289273,400021458
4,https://www.fifa.com/en/match-centre/match/17/...,Haiti 0-1 Scotland,17,285023,289273,400021453


## 13. Test FIFA Match-Centre Data

Test the FIFA match-centre API using the first World Cup 2026
match.

We will inspect the response structure before extracting data
for all 104 matches.

In [38]:
# Test the FIFA match-centre API for the first match

first_match = match_links_df.iloc[0]

stage_id = first_match["StageID"]
match_id = first_match["MatchID"]

match_api_url = (
    f"https://api.fifa.com/api/v3/live/football/"
    f"17/285023/{stage_id}/{match_id}?language=en"
)

match_response = requests.get(
    match_api_url,
    headers=headers,
    timeout=30
)

print("Status code:", match_response.status_code)
print("Content type:", match_response.headers.get("Content-Type"))
print("Response size:", len(match_response.content), "bytes")

Status code: 200
Content type: application/json; charset=utf-8
Response size: 35610 bytes


## 14. Inspect FIFA Match API Structure

Inspect the structure of the FIFA match-centre API response.

We will identify the fields available for teams, scores,
date, stadium, stage, and other match information before
extracting the 104 matches.

In [39]:
# Convert the response to JSON and inspect its structure

match_data = match_response.json()

print("Response type:", type(match_data))

print("\nTop-level keys:")
print(match_data.keys())

print("\nTop-level structure:")
for key, value in match_data.items():
    if isinstance(value, dict):
        print(f"{key}: dictionary")
    elif isinstance(value, list):
        print(f"{key}: list ({len(value)} items)")
    else:
        print(f"{key}: {type(value).__name__}")

Response type: <class 'dict'>

Top-level keys:
dict_keys(['IdMatch', 'IdStage', 'IdGroup', 'IdSeason', 'CoverageLevel', 'IdCompetition', 'CompetitionName', 'SeasonName', 'SeasonShortName', 'Stadium', 'ResultType', 'MatchDay', 'MatchNumber', 'HomeTeamPenaltyScore', 'AwayTeamPenaltyScore', 'AggregateHomeTeamScore', 'AggregateAwayTeamScore', 'Weather', 'Attendance', 'Date', 'LocalDate', 'MatchTime', 'SecondHalfTime', 'FirstHalfTime', 'FirstHalfExtraTime', 'SecondHalfExtraTime', 'Winner', 'Period', 'HomeTeam', 'AwayTeam', 'BallPossession', 'TerritorialPossesion', 'TerritorialThirdPossesion', 'Officials', 'MatchStatus', 'GroupName', 'StageName', 'OfficialityStatus', 'TimeDefined', 'Properties', 'IsUpdateable'])

Top-level structure:
IdMatch: str
IdStage: str
IdGroup: str
IdSeason: str
CoverageLevel: NoneType
IdCompetition: str
CompetitionName: list (1 items)
SeasonName: list (1 items)
SeasonShortName: list (0 items)
Stadium: dictionary
ResultType: int
MatchDay: NoneType
MatchNumber: int
Hom

## 15. Inspect Match Details

Inspect the FIFA API structures for the teams, stadium,
winner, score information, date, stage, and group.

This will allow us to determine exactly how each field
should be mapped to our SQL `Matches` table.

In [41]:
# Inspect nested fields separately to avoid output truncation

print("HOME TEAM")
print(match_data["HomeTeam"])

print("\n" + "=" * 70)
print("AWAY TEAM")
print(match_data["AwayTeam"])

print("\n" + "=" * 70)
print("STADIUM")
print(match_data["Stadium"])

print("\n" + "=" * 70)
print("WINNER")
print(match_data["Winner"])

print("\n" + "=" * 70)
print("RESULT TYPE")
print(match_data["ResultType"])

print("\n" + "=" * 70)
print("MATCH STATUS")
print(match_data["MatchStatus"])

print("\n" + "=" * 70)
print("GROUP NAME")
print(match_data["GroupName"])

print("\n" + "=" * 70)
print("STAGE NAME")
print(match_data["StageName"])

HOME TEAM
{'Score': 2, 'Side': None, 'IdTeam': '43911', 'PictureUrl': 'https://api.fifa.com/api/v3/picture/flags-{format}-{size}/MEX', 'IdCountry': 'MEX', 'TeamType': 1, 'AgeType': 7, 'Tactics': '4-1-2-3', 'TeamName': [{'Locale': 'en-GB', 'Description': 'Mexico'}], 'Abbreviation': 'MEX', 'Coaches': [{'IdCoach': '178119', 'IdCountry': 'MEX', 'PictureUrl': None, 'Name': [{'Locale': 'en-GB', 'Description': 'Rafael MARQUEZ'}], 'Alias': [{'Locale': 'en-GB', 'Description': 'Rafael MARQUEZ'}], 'Role': 1, 'SpecialStatus': None}, {'IdCoach': '45078', 'IdCountry': 'MEX', 'PictureUrl': 'https://digitalhub.fifa.com/transform/b36d8467-71b2-45e2-a681-dfb7e80a9ada/AGUIRRE-Javier_45078', 'Name': [{'Locale': 'en-GB', 'Description': 'Javier Aguirre Onaindía'}], 'Alias': [{'Locale': 'en-GB', 'Description': 'Javier AGUIRRE'}], 'Role': 0, 'SpecialStatus': None}, {'IdCoach': '439543', 'IdCountry': 'ESP', 'PictureUrl': None, 'Name': [{'Locale': 'en-GB', 'Description': 'Antonio Amor Fernández'}], 'Alias': [{'

## 16. Verify FIFA Score and Penalty Handling

Inspect the score fields and penalty fields for a knockout
match that went to a penalty shootout.

This ensures that the `Matches` table stores football goals
correctly and does not incorrectly treat penalty kicks as goals.

In [42]:
# Find a match whose FIFA article text indicates a penalty shootout

penalty_match = match_links_df[
    match_links_df["text"].str.contains("4-6", na=False)
].iloc[0]

print("Match text:", penalty_match["text"])
print("Match ID:", penalty_match["MatchID"])
print("Stage ID:", penalty_match["StageID"])

penalty_api_url = (
    f"https://api.fifa.com/api/v3/live/football/"
    f"17/285023/{penalty_match['StageID']}/{penalty_match['MatchID']}"
    f"?language=en"
)

penalty_response = requests.get(
    penalty_api_url,
    headers=headers,
    timeout=30
)

print("\nStatus code:", penalty_response.status_code)

penalty_data = penalty_response.json()

print("\nHome team:", penalty_data["HomeTeam"]["TeamName"])
print("Home score:", penalty_data["HomeTeam"]["Score"])
print("Home penalty score:", penalty_data["HomeTeamPenaltyScore"])

print("\nAway team:", penalty_data["AwayTeam"]["TeamName"])
print("Away score:", penalty_data["AwayTeam"]["Score"])
print("Away penalty score:", penalty_data["AwayTeamPenaltyScore"])

print("\nWinner:", penalty_data["Winner"])

Match text: France 4-6 England
Match ID: 400021542
Stage ID: 289291

Status code: 200

Home team: [{'Locale': 'en-GB', 'Description': 'France'}]
Home score: 4
Home penalty score: None

Away team: [{'Locale': 'en-GB', 'Description': 'England'}]
Away score: 6
Away penalty score: None

Winner: 43942


### 16.1 Inspect Goal Events

Inspect the goal events for the France vs England match.

We will compare the goal-event count with the score shown by
the FIFA API to determine how actual football goals should be
stored when a match is decided by penalties.

In [43]:
# Inspect the actual goal events for France vs England

print("France goals:")
for goal in penalty_data["HomeTeam"]["Goals"]:
    print(goal)

print("\nEngland goals:")
for goal in penalty_data["AwayTeam"]["Goals"]:
    print(goal)

print("\nNumber of France goal events:",
      len(penalty_data["HomeTeam"]["Goals"]))

print("Number of England goal events:",
      len(penalty_data["AwayTeam"]["Goals"]))

France goals:
{'Type': 2, 'IdPlayer': '389867', 'Minute': "48'", 'IdAssistPlayer': None, 'Period': 5, 'IdGoal': None, 'IdTeam': '43946'}
{'Type': 2, 'IdPlayer': '484860', 'Minute': "54'", 'IdAssistPlayer': None, 'Period': 5, 'IdGoal': None, 'IdTeam': '43946'}
{'Type': 2, 'IdPlayer': '389867', 'Minute': "66'", 'IdAssistPlayer': None, 'Period': 5, 'IdGoal': None, 'IdTeam': '43946'}
{'Type': 2, 'IdPlayer': '398680', 'Minute': "90'+6'", 'IdAssistPlayer': None, 'Period': 5, 'IdGoal': None, 'IdTeam': '43946'}

England goals:
{'Type': 2, 'IdPlayer': '433097', 'Minute': "3'", 'IdAssistPlayer': None, 'Period': 3, 'IdGoal': None, 'IdTeam': '43942'}
{'Type': 2, 'IdPlayer': '403049', 'Minute': "18'", 'IdAssistPlayer': None, 'Period': 3, 'IdGoal': None, 'IdTeam': '43942'}
{'Type': 2, 'IdPlayer': '448196', 'Minute': "37'", 'IdAssistPlayer': None, 'Period': 3, 'IdGoal': None, 'IdTeam': '43942'}
{'Type': 2, 'IdPlayer': '448196', 'Minute': "45'+1'", 'IdAssistPlayer': None, 'Period': 3, 'IdGoal': None, 

## 17. Build One Match Record

Create one clean match record from the FIFA API response.

The extracted fields will correspond to the SQL `Matches`
table.

We will validate the first match before processing all
104 matches.

In [44]:
# Build a clean record for Match 1

home_team = match_data["HomeTeam"]
away_team = match_data["AwayTeam"]

team1_goals = home_team["Score"]
team2_goals = away_team["Score"]

winner_id = match_data["Winner"]

# Determine the result from Team 1's perspective
if winner_id is None:
    result = "D"
elif str(winner_id) == str(home_team["IdTeam"]):
    result = "W"
else:
    result = "L"

match_record = {
    "MatchID": int(match_data["MatchNumber"]),
    "MatchDate": match_data["Date"],
    "Stage": match_data["StageName"][0]["Description"],
    "GroupName": (
        match_data["GroupName"][0]["Description"]
        if match_data["GroupName"]
        else None
    ),
    "StadiumID": match_data["Stadium"]["IdStadium"],
    "Team1ID": home_team["IdTeam"],
    "Team2ID": away_team["IdTeam"],
    "Team1Goals": team1_goals,
    "Team2Goals": team2_goals,
    "WinnerTeamID": winner_id,
    "Result": result
}

print("Clean Match Record:")
print(match_record)

Clean Match Record:
{'MatchID': 1, 'MatchDate': '2026-06-11T19:00:00Z', 'Stage': 'First Stage', 'GroupName': 'Group A', 'StadiumID': '400222084', 'Team1ID': '43911', 'Team2ID': '43883', 'Team1Goals': 2, 'Team2Goals': 0, 'WinnerTeamID': '43911', 'Result': 'W'}


## 18. Extract All 104 FIFA Matches

Retrieve the structured match data for all 104 FIFA World Cup
2026 matches using the official FIFA Match-Centre API.

The data will be stored in a raw DataFrame for validation
before mapping FIFA IDs to our own SQL IDs.

In [45]:
# Retrieve all 104 World Cup 2026 match records

all_match_records = []

for index, row in match_links_df.iterrows():

    stage_id = row["StageID"]
    match_id = row["MatchID"]

    url = (
        f"https://api.fifa.com/api/v3/live/football/"
        f"17/285023/{stage_id}/{match_id}?language=en"
    )

    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    if response.status_code != 200:
        print(
            f"ERROR - Match {index + 1}: "
            f"HTTP {response.status_code}"
        )
        continue

    data = response.json()

    home_team = data["HomeTeam"]
    away_team = data["AwayTeam"]

    winner_id = data["Winner"]

    if winner_id is None:
        result = "D"
    elif str(winner_id) == str(home_team["IdTeam"]):
        result = "W"
    else:
        result = "L"

    record = {
        "MatchNumber": data["MatchNumber"],
        "FIFAMatchID": data["IdMatch"],
        "MatchDate": data["Date"],
        "Stage": (
            data["StageName"][0]["Description"]
            if data["StageName"]
            else None
        ),
        "GroupName": (
            data["GroupName"][0]["Description"]
            if data["GroupName"]
            else None
        ),
        "FIFAStadiumID": data["Stadium"]["IdStadium"],
        "StadiumName": (
            data["Stadium"]["Name"][0]["Description"]
            if data["Stadium"]["Name"]
            else None
        ),
        "Team1FIFAID": home_team["IdTeam"],
        "Team1Name": home_team["TeamName"][0]["Description"],
        "Team2FIFAID": away_team["IdTeam"],
        "Team2Name": away_team["TeamName"][0]["Description"],
        "Team1Goals": home_team["Score"],
        "Team2Goals": away_team["Score"],
        "WinnerFIFAID": winner_id,
        "Result": result
    }

    all_match_records.append(record)

print("Matches successfully collected:", len(all_match_records))

Matches successfully collected: 104


## 19. Validate the Raw Matches Dataset

Validate the collected FIFA World Cup 2026 match records.

We will check:

- Exactly 104 matches
- Match numbers 1–104
- No duplicate matches
- No missing team information
- No missing stadium information
- No missing scores
- No missing match dates

In [46]:
# Convert the collected records into a DataFrame

matches_raw_df = pd.DataFrame(all_match_records)

print("Dataset shape:", matches_raw_df.shape)

print("\nMatch numbers:")
print(
    matches_raw_df["MatchNumber"].min(),
    "to",
    matches_raw_df["MatchNumber"].max()
)

print("\nUnique match numbers:",
      matches_raw_df["MatchNumber"].nunique())

print("\nDuplicate Match Numbers:",
      matches_raw_df["MatchNumber"].duplicated().sum())

print("\nMissing values:")
print(matches_raw_df.isna().sum())

Dataset shape: (104, 15)

Match numbers:
1 to 104

Unique match numbers: 104

Duplicate Match Numbers: 0

Missing values:
MatchNumber       0
FIFAMatchID       0
MatchDate         0
Stage             0
GroupName        32
FIFAStadiumID     0
StadiumName       0
Team1FIFAID       0
Team1Name         0
Team2FIFAID       0
Team2Name         0
Team1Goals        0
Team2Goals        0
WinnerFIFAID     20
Result            0
dtype: int64


### 19.1 Investigate Missing Winners

Inspect matches where FIFA does not provide a WinnerFIFAID.

Determine whether these matches were genuine draws or whether
the winner information is represented differently, such as
through a penalty shootout.

In [47]:
# Inspect matches with no FIFA winner

missing_winner_df = matches_raw_df[
    matches_raw_df["WinnerFIFAID"].isna()
][
    [
        "MatchNumber",
        "FIFAMatchID",
        "MatchDate",
        "Stage",
        "GroupName",
        "Team1Name",
        "Team2Name",
        "Team1Goals",
        "Team2Goals",
        "WinnerFIFAID",
        "Result"
    ]
]

print("Matches with missing WinnerFIFAID:",
      len(missing_winner_df))

display(missing_winner_df)

Matches with missing WinnerFIFAID: 20


,MatchNumber,FIFAMatchID,MatchDate,Stage,GroupName,Team1Name,Team2Name,Team1Goals,Team2Goals,WinnerFIFAID,Result
2,3,400021449,2026-06-12T19:00:00Z,First Stage,Group B,Canada,Bosnia and Herzegovina,1,1,None,D
6,7,400021456,2026-06-13T22:00:00Z,First Stage,Group C,Brazil,Morocco,1,1,None,D
7,8,400021447,2026-06-13T19:00:00Z,First Stage,Group B,Qatar,Switzerland,1,1,None,D
10,11,400021470,2026-06-14T20:00:00Z,First Stage,Group F,Netherlands,Japan,2,2,None,D
12,13,400021486,2026-06-15T22:00:00Z,First Stage,Group H,Saudi Arabia,Uruguay,1,1,None,D
13,14,400021482,2026-06-15T16:00:00Z,First Stage,Group H,Spain,Cabo Verde,0,0,None,D
14,15,400021476,2026-06-16T01:00:00Z,First Stage,Group G,IR Iran,New Zealand,2,2,None,D
15,16,400021478,2026-06-15T19:00:00Z,First Stage,Group G,Belgium,Egypt,1,1,None,D
22,23,400021502,2026-06-17T17:00:00Z,First Stage,Group K,Portugal,Congo DR,1,1,None,D
24,25,400021440,2026-06-18T16:00:00Z,First Stage,Group A,Czechia,South Africa,1,1,None,D


## 20. Map FIFA Team IDs to Our Teams Dataset

Compare the FIFA team IDs used in the match data with the
country codes stored in our Teams dataset.

The FIFA IDs are source identifiers. Our own TeamID values
will be used as the foreign keys in the SQL database.

In [49]:
# Load our previously created Teams dataset

from pathlib import Path
import pandas as pd

data_folder = Path.cwd().parent / "Data"

teams_df = pd.read_csv(data_folder / "Teams.csv")

print("Teams dataset shape:", teams_df.shape)
print("\nTeams columns:")
print(teams_df.columns.tolist())

display(teams_df.head())

Teams dataset shape: (48, 6)

Teams columns:
['TeamID', 'TeamName', 'CountryCode', 'Confederation', 'GroupName', 'HostCountry']


,TeamID,TeamName,CountryCode,Confederation,GroupName,HostCountry
0,1,Argentina,ARG,CONMEBOL,Group J,NaN
1,2,Belgium,BEL,UEFA,Group G,NaN
2,3,Brazil,BRA,CONMEBOL,Group C,NaN
3,4,Colombia,COL,CONMEBOL,Group K,NaN
4,5,England,ENG,UEFA,Group L,NaN


## 21. Create FIFA Team ID Mapping

Create a mapping between FIFA's team identifiers and our own
TeamID values.

The mapping will use the FIFA country code (`IdCountry`) and
our `CountryCode` field.

This mapping will later convert the FIFA team IDs in the Matches
dataset into the foreign keys required by the SQL database.

In [50]:
# Build FIFA Team ID mappings from the collected match data

fifa_team_mapping = {}

for _, row in matches_raw_df.iterrows():

    # Team 1
    fifa_team_mapping[row["Team1FIFAID"]] = {
        "CountryCode": None,
        "TeamName": row["Team1Name"]
    }

    # Team 2
    fifa_team_mapping[row["Team2FIFAID"]] = {
        "CountryCode": None,
        "TeamName": row["Team2Name"]
    }

print("Unique FIFA Team IDs found:",
      len(fifa_team_mapping))

print("\nFirst 10 FIFA Team IDs:")
for fifa_id, info in list(fifa_team_mapping.items())[:10]:
    print(fifa_id, "->", info["TeamName"])

Unique FIFA Team IDs found: 48

First 10 FIFA Team IDs:
43911 -> Mexico
43883 -> South Africa
43822 -> Korea Republic
43995 -> Czechia
43899 -> Canada
44037 -> Bosnia and Herzegovina
43921 -> USA
43928 -> Paraguay
43908 -> Haiti
43967 -> Scotland


## 22. Create FIFA-to-SQL Team Mapping

Collect the FIFA country codes for all teams and map them to
our own TeamID values from Teams.csv.

The country code is used as the reliable connection between
FIFA's team data and our SQL Teams table.

In [52]:
# Create a mapping between FIFA Team IDs and our TeamID values

fifa_team_mapping = []

for _, row in matches_raw_df.iterrows():

    # Find the corresponding FIFA match URL
    match_row = match_links_df[
        match_links_df["MatchID"] == int(row["FIFAMatchID"])
    ].iloc[0]

    stage_id = int(match_row["StageID"])
    match_id = int(match_row["MatchID"])

    url = (
        f"https://api.fifa.com/api/v3/live/football/"
        f"17/285023/{stage_id}/{match_id}?language=en"
    )

    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    data = response.json()

    home = data["HomeTeam"]
    away = data["AwayTeam"]

    fifa_team_mapping.append({
        "FIFATeamID": home["IdTeam"],
        "CountryCode": home["IdCountry"],
        "TeamName": home["TeamName"][0]["Description"]
    })

    fifa_team_mapping.append({
        "FIFATeamID": away["IdTeam"],
        "CountryCode": away["IdCountry"],
        "TeamName": away["TeamName"][0]["Description"]
    })

# Remove duplicate teams
fifa_team_mapping_df = (
    pd.DataFrame(fifa_team_mapping)
    .drop_duplicates(subset=["FIFATeamID"])
    .reset_index(drop=True)
)

print("Unique FIFA teams:", len(fifa_team_mapping_df))

display(fifa_team_mapping_df.head(10))

Unique FIFA teams: 48


,FIFATeamID,CountryCode,TeamName
0,43911,MEX,Mexico
1,43883,RSA,South Africa
2,43822,KOR,Korea Republic
3,43995,CZE,Czechia
4,43899,CAN,Canada
5,44037,BIH,Bosnia and Herzegovina
6,43921,USA,USA
7,43928,PAR,Paraguay
8,43908,HAI,Haiti
9,43967,SCO,Scotland


## 23. Create Final TeamID Mapping

Merge the FIFA team mapping with our Teams.csv dataset
using the country code.

This converts FIFA team identifiers into the TeamID values
used as foreign keys in our SQL database.

In [53]:
# Merge FIFA team IDs with our own TeamID values

team_id_mapping_df = fifa_team_mapping_df.merge(
    teams_df[["TeamID", "TeamName", "CountryCode"]],
    on="CountryCode",
    how="left",
    suffixes=("_FIFA", "_SQL")
)

print("Rows:", len(team_id_mapping_df))

print("\nMissing SQL TeamIDs:",
      team_id_mapping_df["TeamID"].isna().sum())

display(team_id_mapping_df.head(10))

Rows: 48

Missing SQL TeamIDs: 0


,FIFATeamID,CountryCode,TeamName_FIFA,TeamID,TeamName_SQL
0,43911,MEX,Mexico,8,Mexico
1,43883,RSA,South Africa,24,South Africa
2,43822,KOR,Korea Republic,31,Korea Republic
3,43995,CZE,Czechia,38,Czechia
4,43899,CAN,Canada,16,Canada
5,44037,BIH,Bosnia and Herzegovina,26,Bosnia and Herzegovina
6,43921,USA,USA,12,USA
7,43928,PAR,Paraguay,32,Paraguay
8,43908,HAI,Haiti,39,Haiti
9,43967,SCO,Scotland,33,Scotland


## 24. Map FIFA Team IDs to SQL TeamIDs

Replace the FIFA team identifiers in the raw matches dataset
with the TeamID values used by our SQL Teams table.

The FIFA identifiers will be kept temporarily for validation.

In [54]:
# Create lookup dictionaries

fifa_to_sql_team = dict(
    zip(
        team_id_mapping_df["FIFATeamID"],
        team_id_mapping_df["TeamID"]
    )
)

# Map FIFA IDs to our SQL TeamIDs

matches_raw_df["Team1ID"] = (
    matches_raw_df["Team1FIFAID"]
    .map(fifa_to_sql_team)
)

matches_raw_df["Team2ID"] = (
    matches_raw_df["Team2FIFAID"]
    .map(fifa_to_sql_team)
)

matches_raw_df["WinnerTeamID"] = (
    matches_raw_df["WinnerFIFAID"]
    .map(fifa_to_sql_team)
)

print("Missing Team1ID:",
      matches_raw_df["Team1ID"].isna().sum())

print("Missing Team2ID:",
      matches_raw_df["Team2ID"].isna().sum())

print("Missing WinnerTeamID:",
      matches_raw_df["WinnerTeamID"].isna().sum())

display(
    matches_raw_df[
        [
            "MatchNumber",
            "Team1Name",
            "Team1ID",
            "Team2Name",
            "Team2ID",
            "WinnerFIFAID",
            "WinnerTeamID"
        ]
    ].head(10)
)

Missing Team1ID: 0
Missing Team2ID: 0
Missing WinnerTeamID: 20


,MatchNumber,Team1Name,Team1ID,Team2Name,Team2ID,WinnerFIFAID,WinnerTeamID
0,1,Mexico,8,South Africa,24,43911,8.0
1,2,Korea Republic,31,Czechia,38,43822,31.0
2,3,Canada,16,Bosnia and Herzegovina,26,None,NaN
3,4,USA,12,Paraguay,32,43921,12.0
4,5,Haiti,39,Scotland,33,43967,33.0
5,6,Australia,13,Türkiye,47,43976,13.0
6,7,Brazil,3,Morocco,21,None,NaN
7,8,Qatar,44,Switzerland,11,None,NaN
8,9,Côte d'Ivoire,17,Ecuador,28,43854,17.0
9,10,Germany,7,Curaçao,37,43948,7.0


## 25. Map FIFA Stadium IDs to SQL StadiumID

Create a mapping between FIFA stadium identifiers and our
own StadiumID values.

The stadium name will be used to connect FIFA's match data
with the Stadiums.csv dataset.

In [55]:
# Load our Stadiums dataset

stadiums_df = pd.read_csv(
    data_folder / "Stadiums.csv"
)

print("Stadiums dataset shape:", stadiums_df.shape)

print("\nStadiums columns:")
print(stadiums_df.columns.tolist())

display(stadiums_df.head(16))

Stadiums dataset shape: (16, 5)

Stadiums columns:
['StadiumID', 'StadiumName', 'City', 'Country', 'Capacity']


,StadiumID,StadiumName,City,Country,Capacity
0,1,Toronto Stadium,"Toronto, Ontario",Canada,43036
1,2,BC Place Vancouver,"Vancouver, British Colombia",Canada,52497
2,3,Mexico City Stadium,Mexico City,Mexico,80824
3,4,Guadalajara Stadium,"Zapopan, Jalisco",Mexico,45664
4,5,Estadio Monterrey,"Guadalupe, Nuevo Leon",Mexico,51243
5,6,Atlanta Stadium,"Atlanta, Georgia",USA,68239
6,7,Boston Stadium,"Foxborough, Massachusetts",USA,64146
7,8,Dallas Stadium,"Arlington, Texas",USA,70649
8,9,Houston Stadium,"Houston, Texas",USA,68777
9,10,Kansas City Stadium,"Kansas City, Missouri",USA,69045


## 26. Create FIFA-to-SQL Stadium Mapping

Create a mapping between FIFA stadium identifiers and our
own StadiumID values.

The mapping will use the stadium name returned by FIFA and
the stadium name stored in Stadiums.csv.

In [56]:
# Collect FIFA stadium IDs and names from the 104 matches

fifa_stadium_mapping = (
    matches_raw_df[
        ["FIFAStadiumID", "StadiumName"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Unique FIFA stadiums found:",
      len(fifa_stadium_mapping))

display(fifa_stadium_mapping)

Unique FIFA stadiums found: 16


,FIFAStadiumID,StadiumName
0,400222084,Mexico City Stadium
1,400252150,Guadalajara Stadium
2,400242032,Toronto Stadium
3,400017978,Los Angeles Stadium
4,400248623,Boston Stadium
5,400248370,BC Place Vancouver
6,400257536,New York/New Jersey Stadium
7,400257521,San Francisco Bay Area Stadium
8,400248622,Philadelphia Stadium
9,400249385,Houston Stadium


## 27. Create Final StadiumID Mapping

Merge the FIFA stadium identifiers with our Stadiums.csv
dataset using the stadium name.

This converts FIFA stadium IDs into the StadiumID values
used as foreign keys in our SQL Matches table.

In [57]:
# Merge FIFA stadium IDs with our own StadiumID values

stadium_id_mapping_df = fifa_stadium_mapping.merge(
    stadiums_df[["StadiumID", "StadiumName"]],
    on="StadiumName",
    how="left"
)

print("Rows:", len(stadium_id_mapping_df))

print(
    "Missing SQL StadiumIDs:",
    stadium_id_mapping_df["StadiumID"].isna().sum()
)

display(stadium_id_mapping_df)

Rows: 16
Missing SQL StadiumIDs: 2


,FIFAStadiumID,StadiumName,StadiumID
0,400222084,Mexico City Stadium,3.0
1,400252150,Guadalajara Stadium,4.0
2,400242032,Toronto Stadium,1.0
3,400017978,Los Angeles Stadium,11.0
4,400248623,Boston Stadium,7.0
5,400248370,BC Place Vancouver,2.0
6,400257536,New York/New Jersey Stadium,NaN
7,400257521,San Francisco Bay Area Stadium,15.0
8,400248622,Philadelphia Stadium,14.0
9,400249385,Houston Stadium,9.0


### 27.1 Investigate Missing Stadium Mappings

Identify the FIFA stadium names that did not match the
stadium names in Stadiums.csv.

We will inspect the exact spelling before deciding how
to correct the mapping.

In [58]:
# Show stadiums that could not be mapped

missing_stadiums = stadium_id_mapping_df[
    stadium_id_mapping_df["StadiumID"].isna()
]

print("Missing stadium mappings:")
display(missing_stadiums)

print("\nAll FIFA stadium names:")
print(fifa_stadium_mapping["StadiumName"].tolist())

print("\nOur stadium names:")
print(stadiums_df["StadiumName"].tolist())

Missing stadium mappings:


,FIFAStadiumID,StadiumName,StadiumID
6,400257536,New York/New Jersey Stadium,NaN
11,400238450,Monterrey Stadium,NaN



All FIFA stadium names:
['Mexico City Stadium', 'Guadalajara Stadium', 'Toronto Stadium', 'Los Angeles Stadium', 'Boston Stadium', 'BC Place Vancouver', 'New York/New Jersey Stadium', 'San Francisco Bay Area Stadium', 'Philadelphia Stadium', 'Houston Stadium', 'Dallas Stadium', 'Monterrey Stadium', 'Miami Stadium', 'Atlanta Stadium', 'Seattle Stadium', 'Kansas City Stadium']

Our stadium names:
['Toronto Stadium', 'BC Place Vancouver', 'Mexico City Stadium', 'Guadalajara Stadium', 'Estadio Monterrey', 'Atlanta Stadium', 'Boston Stadium', 'Dallas Stadium', 'Houston Stadium', 'Kansas City Stadium', 'Los Angeles Stadium', 'Miami Stadium', 'New York New Jersey Stadium', 'Philadelphia Stadium', 'San Francisco Bay Area Stadium', 'Seattle Stadium']


### 27.2 Resolve Stadium Name Differences

Resolve the two verified naming differences between FIFA's
match data and our Stadiums.csv dataset.

The original FIFA stadium names will remain unchanged.
A mapping dictionary will connect them to our own StadiumID values.

In [59]:
# Resolve the two verified FIFA stadium name differences

stadium_name_mapping = {
    "New York/New Jersey Stadium": "New York New Jersey Stadium",
    "Monterrey Stadium": "Estadio Monterrey"
}

# Create a standardized stadium name only for mapping
fifa_stadium_mapping["MappedStadiumName"] = (
    fifa_stadium_mapping["StadiumName"]
    .replace(stadium_name_mapping)
)

# Merge with our Stadiums dataset
stadium_id_mapping_df = fifa_stadium_mapping.merge(
    stadiums_df[["StadiumID", "StadiumName"]],
    left_on="MappedStadiumName",
    right_on="StadiumName",
    how="left"
)

print("Rows:", len(stadium_id_mapping_df))

print(
    "Missing SQL StadiumIDs:",
    stadium_id_mapping_df["StadiumID"].isna().sum()
)

display(
    stadium_id_mapping_df[
        [
            "FIFAStadiumID",
            "StadiumName_x",
            "MappedStadiumName",
            "StadiumID"
        ]
    ]
)

Rows: 16
Missing SQL StadiumIDs: 0


,FIFAStadiumID,StadiumName_x,MappedStadiumName,StadiumID
0,400222084,Mexico City Stadium,Mexico City Stadium,3
1,400252150,Guadalajara Stadium,Guadalajara Stadium,4
2,400242032,Toronto Stadium,Toronto Stadium,1
3,400017978,Los Angeles Stadium,Los Angeles Stadium,11
4,400248623,Boston Stadium,Boston Stadium,7
5,400248370,BC Place Vancouver,BC Place Vancouver,2
6,400257536,New York/New Jersey Stadium,New York New Jersey Stadium,13
7,400257521,San Francisco Bay Area Stadium,San Francisco Bay Area Stadium,15
8,400248622,Philadelphia Stadium,Philadelphia Stadium,14
9,400249385,Houston Stadium,Houston Stadium,9


## 28. Map FIFA Stadium IDs to SQL StadiumID

Add the StadiumID values from our Stadiums dataset to the
raw FIFA matches dataset.

The resulting StadiumID values will be used as foreign keys
in the SQL Matches table.

In [60]:
# Create FIFA Stadium ID -> SQL StadiumID lookup

fifa_to_sql_stadium = dict(
    zip(
        stadium_id_mapping_df["FIFAStadiumID"],
        stadium_id_mapping_df["StadiumID"]
    )
)

# Map FIFA stadium IDs to our SQL StadiumID

matches_raw_df["StadiumID"] = (
    matches_raw_df["FIFAStadiumID"]
    .map(fifa_to_sql_stadium)
)

print(
    "Missing StadiumID:",
    matches_raw_df["StadiumID"].isna().sum()
)

display(
    matches_raw_df[
        [
            "MatchNumber",
            "StadiumName",
            "FIFAStadiumID",
            "StadiumID"
        ]
    ].head(10)
)

Missing StadiumID: 0


,MatchNumber,StadiumName,FIFAStadiumID,StadiumID
0,1,Mexico City Stadium,400222084,3
1,2,Guadalajara Stadium,400252150,4
2,3,Toronto Stadium,400242032,1
3,4,Los Angeles Stadium,400017978,11
4,5,Boston Stadium,400248623,7
5,6,BC Place Vancouver,400248370,2
6,7,New York/New Jersey Stadium,400257536,13
7,8,San Francisco Bay Area Stadium,400257521,15
8,9,Philadelphia Stadium,400248622,14
9,10,Houston Stadium,400249385,9


## 29. Validate Match Stages and Groups

Inspect the stage and group values returned by FIFA
before preparing the final Matches dataset.

This ensures that the tournament stages and group information
are correctly represented.

In [61]:
# Inspect unique tournament stages

print("Unique Stages:")
display(
    matches_raw_df["Stage"]
    .value_counts()
    .rename_axis("Stage")
    .reset_index(name="MatchCount")
)

print("\nUnique Group Names:")
display(
    matches_raw_df["GroupName"]
    .value_counts(dropna=False)
    .rename_axis("GroupName")
    .reset_index(name="MatchCount")
)

Unique Stages:


,Stage,MatchCount
0,First Stage,72
1,Round of 32,16
2,Round of 16,8
3,Quarter-final,4
4,Semi-final,2
5,Bronze final,1
6,Final,1



Unique Group Names:


,GroupName,MatchCount
0,None,32
1,Group B,6
2,Group D,6
3,Group C,6
4,Group A,6
5,Group E,6
6,Group F,6
7,Group G,6
8,Group H,6
9,Group I,6


## 30. Validate Match Dates

Compare the FIFA match Date and LocalDate values.

This determines which date should be stored in the SQL Matches table,
which contains a DATE field without a time component.

In [62]:
# Compare FIFA UTC Date with local match date

date_comparison = []

for _, row in match_links_df.iterrows():

    stage_id = row["StageID"]
    match_id = row["MatchID"]

    url = (
        f"https://api.fifa.com/api/v3/live/football/17/285023/"
        f"{stage_id}/{match_id}?language=en"
    )

    response = requests.get(url, headers=headers, timeout=30)
    data = response.json()

    date_comparison.append({
        "MatchNumber": data["MatchNumber"],
        "Date": data["Date"],
        "LocalDate": data["LocalDate"]
    })

date_comparison_df = pd.DataFrame(date_comparison)

date_comparison_df["DateOnly"] = (
    pd.to_datetime(date_comparison_df["Date"])
    .dt.date
)

date_comparison_df["LocalDateOnly"] = (
    pd.to_datetime(date_comparison_df["LocalDate"])
    .dt.date
)

date_comparison_df["DifferentDate"] = (
    date_comparison_df["DateOnly"]
    != date_comparison_df["LocalDateOnly"]
)

print(
    "Matches with different UTC and local dates:",
    date_comparison_df["DifferentDate"].sum()
)

display(
    date_comparison_df[
        date_comparison_df["DifferentDate"]
    ]
)

Matches with different UTC and local dates: 36


,MatchNumber,Date,LocalDate,DateOnly,LocalDateOnly,DifferentDate
1,2,2026-06-12T02:00:00Z,2026-06-11T20:00:00Z,2026-06-12,2026-06-11,True
3,4,2026-06-13T01:00:00Z,2026-06-12T18:00:00Z,2026-06-13,2026-06-12,True
4,5,2026-06-14T01:00:00Z,2026-06-13T21:00:00Z,2026-06-14,2026-06-13,True
5,6,2026-06-14T04:00:00Z,2026-06-13T21:00:00Z,2026-06-14,2026-06-13,True
11,12,2026-06-15T02:00:00Z,2026-06-14T20:00:00Z,2026-06-15,2026-06-14,True
14,15,2026-06-16T01:00:00Z,2026-06-15T18:00:00Z,2026-06-16,2026-06-15,True
18,19,2026-06-17T01:00:00Z,2026-06-16T20:00:00Z,2026-06-17,2026-06-16,True
19,20,2026-06-17T04:00:00Z,2026-06-16T21:00:00Z,2026-06-17,2026-06-16,True
23,24,2026-06-18T02:00:00Z,2026-06-17T20:00:00Z,2026-06-18,2026-06-17,True
27,28,2026-06-19T01:00:00Z,2026-06-18T19:00:00Z,2026-06-19,2026-06-18,True


### 30.1 Verify Local Match Dates

Use FIFA's LocalDate value as the tournament match date.

First, verify that every match has a valid LocalDate
and that there are exactly 104 unique match dates/records.

In [63]:
# Verify LocalDate values for all 104 matches

print("Total matches:", len(date_comparison_df))

print(
    "Missing LocalDate:",
    date_comparison_df["LocalDate"].isna().sum()
)

print(
    "Unique MatchNumbers:",
    date_comparison_df["MatchNumber"].nunique()
)

print(
    "Earliest LocalDate:",
    date_comparison_df["LocalDateOnly"].min()
)

print(
    "Latest LocalDate:",
    date_comparison_df["LocalDateOnly"].max()
)

display(
    date_comparison_df[
        ["MatchNumber", "LocalDate", "LocalDateOnly"]
    ].head(10)
)

Total matches: 104
Missing LocalDate: 0
Unique MatchNumbers: 104
Earliest LocalDate: 2026-06-11
Latest LocalDate: 2026-07-19


,MatchNumber,LocalDate,LocalDateOnly
0,1,2026-06-11T13:00:00Z,2026-06-11
1,2,2026-06-11T20:00:00Z,2026-06-11
2,3,2026-06-12T15:00:00Z,2026-06-12
3,4,2026-06-12T18:00:00Z,2026-06-12
4,5,2026-06-13T21:00:00Z,2026-06-13
5,6,2026-06-13T21:00:00Z,2026-06-13
6,7,2026-06-13T18:00:00Z,2026-06-13
7,8,2026-06-13T12:00:00Z,2026-06-13
8,9,2026-06-14T19:00:00Z,2026-06-14
9,10,2026-06-14T12:00:00Z,2026-06-14


## 31. Prepare MatchDate

Use FIFA's LocalDate as the official match date.

The SQL Matches table stores MatchDate as DATE,
so only the calendar date will be retained.

In [64]:
# Create a lookup of MatchNumber -> LocalDate

local_date_mapping = dict(
    zip(
        date_comparison_df["MatchNumber"],
        date_comparison_df["LocalDateOnly"]
    )
)

# Replace MatchDate with the verified local match date

matches_raw_df["MatchDate"] = (
    matches_raw_df["MatchNumber"]
    .map(local_date_mapping)
)

# Convert to pandas date type

matches_raw_df["MatchDate"] = pd.to_datetime(
    matches_raw_df["MatchDate"]
).dt.date

print("Missing MatchDate:", matches_raw_df["MatchDate"].isna().sum())

display(
    matches_raw_df[
        ["MatchNumber", "MatchDate", "Stage", "GroupName"]
    ].head(10)
)

Missing MatchDate: 0


,MatchNumber,MatchDate,Stage,GroupName
0,1,2026-06-11,First Stage,Group A
1,2,2026-06-11,First Stage,Group A
2,3,2026-06-12,First Stage,Group B
3,4,2026-06-12,First Stage,Group D
4,5,2026-06-13,First Stage,Group C
5,6,2026-06-13,First Stage,Group D
6,7,2026-06-13,First Stage,Group C
7,8,2026-06-13,First Stage,Group B
8,9,2026-06-14,First Stage,Group E
9,10,2026-06-14,First Stage,Group E


## 32. Validate Matches Foreign Keys

Verify that every TeamID and StadiumID in the Matches dataset
exists in our Teams.csv and Stadiums.csv datasets.

This ensures that the final Matches dataset will satisfy
the SQL foreign key constraints.

In [65]:
# Validate TeamID foreign keys

valid_team_ids = set(teams_df["TeamID"])
valid_stadium_ids = set(stadiums_df["StadiumID"])

invalid_team1 = (
    ~matches_raw_df["Team1ID"].isin(valid_team_ids)
).sum()

invalid_team2 = (
    ~matches_raw_df["Team2ID"].isin(valid_team_ids)
).sum()

invalid_winner = (
    matches_raw_df["WinnerTeamID"].notna()
    & ~matches_raw_df["WinnerTeamID"].isin(valid_team_ids)
).sum()

invalid_stadium = (
    ~matches_raw_df["StadiumID"].isin(valid_stadium_ids)
).sum()

print("Invalid Team1ID:", invalid_team1)
print("Invalid Team2ID:", invalid_team2)
print("Invalid WinnerTeamID:", invalid_winner)
print("Invalid StadiumID:", invalid_stadium)

Invalid Team1ID: 0
Invalid Team2ID: 0
Invalid WinnerTeamID: 0
Invalid StadiumID: 0


## 33. Validate Match Integrity

Perform final structural checks on the match records before
preparing the final Matches dataset.

In [66]:
# Match ID validation

match_id_check = (
    matches_raw_df["MatchNumber"].nunique() == 104
    and set(matches_raw_df["MatchNumber"]) == set(range(1, 105))
)

# Team validation

same_team_check = (
    matches_raw_df["Team1ID"] == matches_raw_df["Team2ID"]
).sum()

# Goal validation

negative_goals_check = (
    (matches_raw_df["Team1Goals"] < 0)
    | (matches_raw_df["Team2Goals"] < 0)
).sum()

# Result validation

def expected_result(row):
    if row["WinnerTeamID"] is None or pd.isna(row["WinnerTeamID"]):
        return "D"
    elif row["WinnerTeamID"] == row["Team1ID"]:
        return "W"
    else:
        return "L"

matches_raw_df["ExpectedResult"] = matches_raw_df.apply(
    expected_result,
    axis=1
)

result_mismatch_check = (
    matches_raw_df["Result"]
    != matches_raw_df["ExpectedResult"]
).sum()

print("Total matches:", len(matches_raw_df))
print("Match IDs 1–104 complete:", match_id_check)
print("Teams playing themselves:", same_team_check)
print("Matches with negative goals:", negative_goals_check)
print("Result mismatches:", result_mismatch_check)

Total matches: 104
Match IDs 1–104 complete: True
Teams playing themselves: 0
Matches with negative goals: 0
Result mismatches: 0


## 34. Prepare Final Matches Dataset

Select and order the columns required by the SQL Matches table.

The final dataset will use our own TeamID and StadiumID values
as foreign keys.

In [67]:
# Create the final SQL-ready Matches dataset

matches_df = matches_raw_df[
    [
        "MatchNumber",
        "MatchDate",
        "Stage",
        "GroupName",
        "StadiumID",
        "Team1ID",
        "Team2ID",
        "Team1Goals",
        "Team2Goals",
        "WinnerTeamID",
        "Result"
    ]
].copy()

# Rename MatchNumber to our SQL MatchID

matches_df = matches_df.rename(
    columns={
        "MatchNumber": "MatchID"
    }
)

print("Rows:", len(matches_df))
print("Columns:", list(matches_df.columns))

display(matches_df.head(10))

Rows: 104
Columns: ['MatchID', 'MatchDate', 'Stage', 'GroupName', 'StadiumID', 'Team1ID', 'Team2ID', 'Team1Goals', 'Team2Goals', 'WinnerTeamID', 'Result']


,MatchID,MatchDate,Stage,GroupName,StadiumID,Team1ID,Team2ID,Team1Goals,Team2Goals,WinnerTeamID,Result
0,1,2026-06-11,First Stage,Group A,3,8,24,2,0,8.0,W
1,2,2026-06-11,First Stage,Group A,4,31,38,2,1,31.0,W
2,3,2026-06-12,First Stage,Group B,1,16,26,1,1,NaN,D
3,4,2026-06-12,First Stage,Group D,11,12,32,4,1,12.0,W
4,5,2026-06-13,First Stage,Group C,7,39,33,0,1,33.0,L
5,6,2026-06-13,First Stage,Group D,2,13,47,2,0,13.0,W
6,7,2026-06-13,First Stage,Group C,13,3,21,1,1,NaN,D
7,8,2026-06-13,First Stage,Group B,15,44,11,1,1,NaN,D
8,9,2026-06-14,First Stage,Group E,14,17,28,1,0,17.0,W
9,10,2026-06-14,First Stage,Group E,9,7,37,7,1,7.0,W


## 35. Final Matches Dataset Validation

Perform the final validation of the SQL-ready Matches dataset.

Verify row count, column structure, unique MatchIDs,
required values, score consistency, and draw handling.

In [68]:
# Final validation of the Matches dataset

expected_columns = [
    "MatchID",
    "MatchDate",
    "Stage",
    "GroupName",
    "StadiumID",
    "Team1ID",
    "Team2ID",
    "Team1Goals",
    "Team2Goals",
    "WinnerTeamID",
    "Result"
]

print("Rows:", len(matches_df))
print("Columns correct:", list(matches_df.columns) == expected_columns)
print("Unique MatchIDs:", matches_df["MatchID"].nunique())

print("\nMissing values:")
display(matches_df.isna().sum())

print("\nResult distribution:")
display(matches_df["Result"].value_counts())

print("\nTotal goals:")
print(
    "Team 1 goals:",
    matches_df["Team1Goals"].sum()
)

print(
    "Team 2 goals:",
    matches_df["Team2Goals"].sum()
)

print(
    "Total tournament goals:",
    matches_df["Team1Goals"].sum()
    + matches_df["Team2Goals"].sum()
)

Rows: 104
Columns correct: True
Unique MatchIDs: 104

Missing values:


MatchID          0
MatchDate        0
Stage            0
GroupName       32
StadiumID        0
Team1ID          0
Team2ID          0
Team1Goals       0
Team2Goals       0
WinnerTeamID    20
Result           0
dtype: int64


Result distribution:


Result
W    51
L    33
D    20
Name: count, dtype: int64


Total goals:
Team 1 goals: 179
Team 2 goals: 129
Total tournament goals: 308


## 36. Export Matches Dataset

Export the validated SQL-ready Matches dataset as Matches.csv
inside the project's Data folder.

In [69]:
# Export the final Matches dataset

matches_file = data_folder / "Matches.csv"

matches_df.to_csv(
    matches_file,
    index=False
)

print("Matches.csv exported successfully!")
print("File:", matches_file)
print("Rows:", len(matches_df))

Matches.csv exported successfully!
File: d:\ai course hw\FIFA_WorldCup_2026_Project\Data\Matches.csv
Rows: 104


## 37. Verify Exported Matches CSV

Read the exported Matches.csv file back from the Data folder
and verify that the saved file contains the expected rows,
columns, and MatchIDs.

In [70]:
# Read the exported CSV back into pandas

matches_csv_check = pd.read_csv(
    data_folder / "Matches.csv"
)

print("Rows:", len(matches_csv_check))
print("Columns:", list(matches_csv_check.columns))
print("Unique MatchIDs:", matches_csv_check["MatchID"].nunique())

print("\nFirst 5 rows:")
display(matches_csv_check.head())

Rows: 104
Columns: ['MatchID', 'MatchDate', 'Stage', 'GroupName', 'StadiumID', 'Team1ID', 'Team2ID', 'Team1Goals', 'Team2Goals', 'WinnerTeamID', 'Result']
Unique MatchIDs: 104

First 5 rows:


,MatchID,MatchDate,Stage,GroupName,StadiumID,Team1ID,Team2ID,Team1Goals,Team2Goals,WinnerTeamID,Result
0,1,2026-06-11,First Stage,Group A,3,8,24,2,0,8.0,W
1,2,2026-06-11,First Stage,Group A,4,31,38,2,1,31.0,W
2,3,2026-06-12,First Stage,Group B,1,16,26,1,1,NaN,D
3,4,2026-06-12,First Stage,Group D,11,12,32,4,1,12.0,W
4,5,2026-06-13,First Stage,Group C,7,39,33,0,1,33.0,L
